# ML-Bench: QLoRA iteration loop

Trains **Qwen2.5-7B-Instruct (4-bit)** with rejection-sampling SFT across rounds on the
ML-Bench CPS×ML task suite (t01 Kalman, t02 system identification, t05 fault detection).

Per round:
1. Sample `n_per_task` solutions from the current model.
2. Score every generation through the deterministic sandbox (and optionally Opus).
3. Build a ChatML SFT pool from the top-k positives (+ verified judge corrections).
4. Train QLoRA (r=16, α=32) for 1–3 epochs on that pool.
5. Re-evaluate with the new adapter; log {mean, best, pass-rate} per task.

Runs on Colab L4 (24 GB) cleanly. Free-tier T4 (16 GB) is too tight for 7B QLoRA training
— swap `MODEL_NAME` to `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` if needed.

## 0. Mount Drive + locate the repo

Two workflows are supported:
- **A.** `git clone` your fork into `/content/ML-Bench`.
- **B.** Pull a previously-uploaded copy from Drive (e.g. `/content/drive/MyDrive/ML-Bench`).

Pick one; comment the other.

In [ ]:
import os
from pathlib import Path

REPO = Path('/content/ML-Bench')
GIT_URL = 'https://github.com/boeing23/ML-Bench.git'

if not REPO.exists():
    print(f'cloning {GIT_URL} → {REPO} ...')
    rc = os.system(f'git clone {GIT_URL} {REPO}')
    if rc != 0:
        raise RuntimeError(f'git clone failed (exit {rc})')

os.chdir(REPO)
print('cwd:', os.getcwd())
print('top-level:', sorted(os.listdir(REPO))[:15])


Mounted at /content/drive
cp: cannot stat '/content/drive/MyDrive/ML-Bench': No such file or directory


FileNotFoundError: [Errno 2] No such file or directory: '/content/ML-Bench'

## 1. Install dependencies

Unsloth pins exact versions of `transformers`, `bitsandbytes`, `peft`, `trl`, etc. that are
known to cooperate. Pinning anything stricter here just causes resolver thrash.

In [ ]:
%%capture
!pip install -qU pip
# numpy: satisfy numba (<2.1) and modern packages (>=1.26). Force-reinstall to fix any ABI drift.
!pip install -q --force-reinstall "numpy>=1.26,<2.1"
# bitsandbytes: transformers requires >=0.46.1 for 4-bit quantization
!pip install -q "bitsandbytes>=0.46.1"
# Don't pin trl — unsloth-zoo requires trl>=0.18.2 and pulls a working version itself.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "peft" "accelerate"
!pip install -q anthropic pyyaml pydantic

# IMPORTANT: after this cell finishes, restart the runtime once (Runtime → Restart)
# so the kernel reloads with the freshly-installed numpy + bitsandbytes.
# Then re-run from this cell down.

In [ ]:
# Optional: enable the Opus judge (otherwise the loop runs deterministic-only)
from getpass import getpass
if not os.environ.get('ANTHROPIC_API_KEY'):
    try:
        os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY (blank to skip): ').strip()
    except Exception:
        pass
USE_JUDGE = bool(os.environ.get('ANTHROPIC_API_KEY'))
print('judge enabled:', USE_JUDGE)

## 2. Generate task data (deterministic)

Each script writes `data/` (model-visible) and `hidden/` (judge-only) under each task dir.
Idempotent — safe to re-run.

In [ ]:
!python scripts/generate_task01_data.py
!python scripts/generate_task02_data.py
!python scripts/generate_task05_data.py

## 3. Smoke-test the harness

Before any training, confirm the reference solutions still score as expected
(t01 ≈ 0.588, t02 ≈ 0.890, t05 = 1.000). If anything drifts, fix it before continuing.

In [ ]:
!python -m ml_bench.eval.runner --task t01_kalman_filter         --solution ml_bench/tasks/t01_kalman_filter/reference_solution.py         --no-judge | tail -5
!python -m ml_bench.eval.runner --task t02_system_identification --solution ml_bench/tasks/t02_system_identification/reference_solution.py --no-judge | tail -5
!python -m ml_bench.eval.runner --task t05_fault_detection       --solution ml_bench/tasks/t05_fault_detection/reference_solution.py       --no-judge | tail -5

## 4. Load Qwen2.5-7B (4-bit) + attach LoRA adapters

We attach the LoRA at load time so the same model object can swap between
inference (`FastLanguageModel.for_inference`) and training (`for_training`)
without reload across rounds.

In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 4096
#MODEL_NAME = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'  # T4-fallback

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

## 5. Per-round helpers

Each round of the loop runs four functions in order:
1. `generate_solutions` — Qwen samples N solutions per task.
2. `evaluate_round` — score every generation through the harness; record per-task metrics.
3. `score_and_format` — build the SFT pool (top-k positives + verified corrections).
4. `train_round` — QLoRA SFT on the pool.

In [ ]:
import json
import re
from pathlib import Path

from ml_bench.training.chat_format import SYSTEM_PROMPT

TASKS = ['t01_kalman_filter', 't02_system_identification', 't05_fault_detection']
TASKS_DIR = Path('ml_bench/tasks')
RUNS_DIR = Path('runs')
RUNS_DIR.mkdir(exist_ok=True)

_FENCE_RE = re.compile(r'```(?:python)?\s*\n(.*?)```', re.DOTALL)

def _extract_python(text: str) -> str:
    """Pull the first ```python``` fence; fall back to the raw text."""
    m = _FENCE_RE.search(text)
    return (m.group(1) if m else text).strip()

def generate_solutions(round_num: int, n_per_task: int = 8,
                       max_new_tokens: int = 2048,
                       temperature: float = 0.7, top_p: float = 0.9) -> Path:
    """Sample `n_per_task` solutions per task; write generations.jsonl."""
    FastLanguageModel.for_inference(model)
    out_path = RUNS_DIR / f'round_{round_num}' / 'generations.jsonl'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sample_idx = 0
    with out_path.open('w') as fh:
        for tid in TASKS:
            prompt = (TASKS_DIR / tid / 'prompt.md').read_text()
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': prompt},
            ]
            input_ids = tokenizer.apply_chat_template(
                messages, return_tensors='pt', add_generation_prompt=True,
            ).to(model.device)
            for _ in range(n_per_task):
                gen = model.generate(
                    input_ids,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=temperature,
                    top_p=top_p,
                    pad_token_id=tokenizer.eos_token_id,
                )
                resp = tokenizer.decode(
                    gen[0][input_ids.shape[-1]:], skip_special_tokens=True,
                )
                code = _extract_python(resp)
                fh.write(json.dumps({
                    'round': round_num,
                    'task_id': tid,
                    'sample_idx': sample_idx,
                    'solution_text': code,
                    'generated_by': f'qwen_round_{round_num}',
                }) + '\n')
                sample_idx += 1
    print(f'wrote {sample_idx} generations → {out_path}')
    return out_path

In [ ]:
import yaml
from ml_bench.runtime.sandbox import run_episode

def evaluate_round(round_num: int) -> dict:
    """Score every generation in runs/round_N/generations.jsonl through the harness.
    Returns a per-task summary {mean_score, best_score, pass_rate, n_samples}.
    Also writes runs/round_N/eval_summary.json and runs/round_N/scores.jsonl.
    """
    gens_path = RUNS_DIR / f'round_{round_num}' / 'generations.jsonl'
    scores_path = RUNS_DIR / f'round_{round_num}' / 'scores.jsonl'
    summary_path = RUNS_DIR / f'round_{round_num}' / 'eval_summary.json'
    by_task = {tid: [] for tid in TASKS}
    cfg_cache: dict = {}

    with scores_path.open('w') as out_fh:
        for line in gens_path.open():
            rec = json.loads(line)
            tid = rec['task_id']
            if tid not in cfg_cache:
                cfg_cache[tid] = yaml.safe_load(
                    (TASKS_DIR / tid / 'task.yaml').read_text(),
                ) or {}
            limits = (cfg_cache[tid].get('limits') or {})
            res = run_episode(
                task_dir=TASKS_DIR / tid,
                solution_text=rec['solution_text'],
                wall_clock_s=int(limits.get('wall_clock_s', 120)),
                memory_mb=int(limits.get('memory_mb', 4096)),
            )
            score = float(res.score)
            metrics = (res.judge_payload or {}).get('metrics', {})
            by_task[tid].append(score)
            out_fh.write(json.dumps({
                'round': round_num,
                'task_id': tid,
                'sample_idx': rec['sample_idx'],
                'score': score,
                'metrics': metrics,
                'timed_out': res.timed_out,
            }) + '\n')

    summary = {}
    for tid, scores in by_task.items():
        n = len(scores)
        summary[tid] = {
            'round': round_num,
            'n_samples': n,
            'mean_score': float(sum(scores) / n) if n else 0.0,
            'best_score': float(max(scores)) if scores else 0.0,
            'pass_rate': float(sum(s > 0 for s in scores) / n) if n else 0.0,
        }
    summary_path.write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))
    return summary

In [ ]:
from ml_bench.training.sft_formatter import format_round

def score_and_format(round_num: int, top_k: int = 3,
                     include_corrections: bool = True,
                     use_judge: bool = USE_JUDGE) -> dict:
    """Run the SFT formatter on this round's generations.
    Mixes prior-round SFT records with geometric decay so the pool keeps
    earlier wins instead of catastrophically forgetting them.
    """
    return format_round(
        generations_path=RUNS_DIR / f'round_{round_num}' / 'generations.jsonl',
        output_path=RUNS_DIR / f'round_{round_num}' / 'sft_training.jsonl',
        top_k_per_task=top_k,
        include_corrections=include_corrections,
        use_judge=use_judge,
        mix_from=RUNS_DIR,
        decay=0.5,
    )

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

def _load_sft_dataset(round_num: int) -> Dataset:
    rows = [json.loads(l) for l in (RUNS_DIR / f'round_{round_num}' / 'sft_training.jsonl').open()]
    if not rows:
        raise ValueError(f'no SFT records for round {round_num}')
    texts = [tokenizer.apply_chat_template(r['messages'], tokenize=False) for r in rows]
    return Dataset.from_dict({'text': texts})

def train_round(round_num: int, num_epochs: int = 2,
                lr: float = 2e-4) -> str:
    """QLoRA SFT on runs/round_N/sft_training.jsonl. Saves adapter under adapters/round_N."""
    FastLanguageModel.for_training(model)
    ds = _load_sft_dataset(round_num)
    out_dir = f'adapters/round_{round_num}'
    args = SFTConfig(
        output_dir=out_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=num_epochs,
        learning_rate=lr,
        warmup_ratio=0.1,
        lr_scheduler_type='cosine',
        logging_steps=5,
        save_strategy='epoch',
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field='text',
        packing=False,
        report_to='none',
        seed=42,
    )
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=ds,
        args=args,
    )
    trainer.train()
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    print(f'saved adapter → {out_dir}')
    return out_dir

## 6. Iteration loop + per-round metrics plot

The loop logs `{eval, format}` per round to `runs/history.jsonl` and renders a
3-panel curve (mean / best / pass-rate per task per round). The headline metric
is whether the curves trend up across rounds without the OOD generalization
gap (judged separately via `metrics.out_of_distribution_*` in `scores.jsonl`)
blowing out.

In [ ]:
import matplotlib.pyplot as plt

def run_iteration_loop(num_rounds: int = 3, n_per_task: int = 8,
                       top_k: int = 3, num_epochs: int = 2,
                       use_judge: bool = USE_JUDGE) -> list:
    history_path = RUNS_DIR / 'history.jsonl'
    history: list = []
    for r in range(num_rounds):
        print(f'\n========== ROUND {r} ==========')
        generate_solutions(r, n_per_task=n_per_task)
        eval_summary = evaluate_round(r)
        format_summary = score_and_format(
            r, top_k=top_k, include_corrections=use_judge, use_judge=use_judge,
        )
        # Train only if there's a non-empty pool AND there's another round to follow.
        if format_summary.get('n_total_out', 0) > 0 and r < num_rounds - 1:
            train_round(r, num_epochs=num_epochs)
        record = {'round': r, 'eval': eval_summary, 'format': format_summary}
        history.append(record)
        with history_path.open('a') as hf:
            hf.write(json.dumps(record) + '\n')
    plot_history(history)
    return history

def plot_history(history: list) -> None:
    if not history:
        print('no history to plot'); return
    rounds = [h['round'] for h in history]
    fig, axes = plt.subplots(1, len(TASKS), figsize=(15, 4), sharey=True)
    if len(TASKS) == 1:
        axes = [axes]
    for ax, tid in zip(axes, TASKS):
        means = [h['eval'][tid]['mean_score'] for h in history]
        bests = [h['eval'][tid]['best_score'] for h in history]
        rates = [h['eval'][tid]['pass_rate']  for h in history]
        ax.plot(rounds, means, marker='o', label='mean')
        ax.plot(rounds, bests, marker='s', label='best')
        ax.plot(rounds, rates, marker='^', label='pass rate')
        ax.set_title(tid)
        ax.set_xlabel('round')
        ax.set_ylim(-0.05, 1.05)
        ax.grid(alpha=0.3)
        ax.legend(loc='lower right', fontsize=8)
    axes[0].set_ylabel('score')
    plt.tight_layout()
    out = RUNS_DIR / 'training_curve.png'
    plt.savefig(out, dpi=120)
    plt.show()
    print(f'saved → {out}')

## 7. Run it

Default: 3 rounds, 8 samples per task per round, top-3 positives kept, 2 epochs per train.
Round 0 is a pure baseline (no training has happened yet); rounds 1+ are trained on
the cumulative SFT pool from prior rounds.

In [ ]:
history = run_iteration_loop(
    num_rounds=3,
    n_per_task=8,
    top_k=3,
    num_epochs=2,
    use_judge=USE_JUDGE,
)

## 8. Inspecting OOD generalization

`evaluate_round` writes per-sample `scores.jsonl` containing each task's
`metrics` dict (e.g. `in_distribution_f1` / `out_of_distribution_f1` for t05,
or ID/OOD RMSE for t01/t02). The cell below plots ID vs OOD per round so you
can watch the **gap** — the headline reward-integrity metric — across rounds.

In [ ]:
ID_KEY = {
    't01_kalman_filter': 'id_rmse',
    't02_system_identification': 'id_rmse',
    't05_fault_detection': 'in_distribution_f1',
}
OOD_KEY = {
    't01_kalman_filter': 'ood_rmse',
    't02_system_identification': 'ood_rmse',
    't05_fault_detection': 'out_of_distribution_f1',
}

def plot_id_ood_gap(num_rounds: int) -> None:
    fig, axes = plt.subplots(1, len(TASKS), figsize=(15, 4))
    if len(TASKS) == 1:
        axes = [axes]
    for ax, tid in zip(axes, TASKS):
        id_means, ood_means = [], []
        for r in range(num_rounds):
            scores_path = RUNS_DIR / f'round_{r}' / 'scores.jsonl'
            if not scores_path.exists():
                id_means.append(float('nan')); ood_means.append(float('nan')); continue
            id_vals, ood_vals = [], []
            for line in scores_path.open():
                rec = json.loads(line)
                if rec['task_id'] != tid or rec['score'] <= 0:
                    continue
                m = rec.get('metrics') or {}
                if ID_KEY[tid] in m: id_vals.append(float(m[ID_KEY[tid]]))
                if OOD_KEY[tid] in m: ood_vals.append(float(m[OOD_KEY[tid]]))
            id_means.append(sum(id_vals) / len(id_vals) if id_vals else float('nan'))
            ood_means.append(sum(ood_vals) / len(ood_vals) if ood_vals else float('nan'))
        rounds = list(range(num_rounds))
        ax.plot(rounds, id_means, marker='o', label=f'ID ({ID_KEY[tid]})')
        ax.plot(rounds, ood_means, marker='s', label=f'OOD ({OOD_KEY[tid]})')
        ax.set_title(tid); ax.set_xlabel('round'); ax.grid(alpha=0.3)
        ax.legend(loc='best', fontsize=8)
    plt.tight_layout()
    plt.savefig(RUNS_DIR / 'id_ood_gap.png', dpi=120)
    plt.show()

plot_id_ood_gap(num_rounds=len(history))